In [ ]:
#| default_exp docsprocs

# docsprocs

> A docs-build notebook processor that color-codes cells by boopiter type, so the rendered site echoes the app's colored left bars.

Registered via `doc_procs` in `pyproject.toml [tool.nbdev]`, `color_cells` runs over every notebook during the docs build (before Quarto renders). It wraps each markdown/raw cell's source in a Quarto fenced div classed by its boopiter cell type (`.boop-note` green, `.boop-prompt` red, `.boop-raw` orange); `styles.css` turns those into the colored left bar. Code cells already carry `.cell-code`, so they're colored in CSS alone. The page's H1 title cell is left untouched so Quarto's title handling isn't disturbed.

In [ ]:
#| export
import re

# a Prompt+Reply markdown cell (solveit encoding) carries this separator -- see serialize.py
_SEP_RE = re.compile(r'##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_[0-9a-f]+ -->')

def color_cells(cell):
    "nbdev docs processor: wrap a markdown/raw cell's source in a Quarto fenced div classed by its boopiter cell type, so styles.css can draw boopiter's colored left bar (green note / red prompt / orange raw). Code cells carry `.cell-code` and are colored via CSS; the page-title (H1) cell is left alone."
    t = cell.get('cell_type'); src = cell.get('source') or ''
    if not src.strip(): return
    first = src.lstrip().splitlines()[0]
    if t == 'raw': cls = 'boop-raw'
    elif t == 'markdown':
        if first.startswith('# '): return          # leave the page-title (H1) cell alone
        cls = 'boop-prompt' if _SEP_RE.search(src) else 'boop-note'
    else: return                                    # code cells are handled in CSS
    cell['source'] = f'::: {{.boopcell .{cls}}}\n{src}\n:::'


In [ ]:
#| export
import hashlib, os, shutil, socket, subprocess, tempfile
from pathlib import Path

SHARE_REPO   = 'boops'                              # GitHub repo whose Pages site serves the rendered notebooks
SHARE_CLONE  = Path.home()/'.boopiter'/'boops'      # local working clone, created on first share
_QUARTO_YML  = """project:
  type: default
format:
  html:
    embed-resources: true
    theme:
      dark: [cosmo, booptheme.scss]
      light: cosmo
    highlight-style:
      light: arrow
      dark: dracula
    css: styles.css
    toc: true
execute:
  enabled: false
"""

def share_slug(path) -> str:
    "Six hex chars identifying a notebook by machine and location: sha256 of hostname + its canonical path. Deterministic, so re-sharing the same file overwrites the same page and a link you've already sent stays current -- that's the whole point of not using a random suffix. Keyed on more than the basename because 'example.ipynb' in two checkouts, or on two machines, are different documents that would otherwise fight over one URL. realpath (not abspath) so two symlinked routes to one file still collapse to a single page, and a NUL separator so host 'a' + path 'b/c' can't collide with host 'a/b' + path 'c'."
    key = f"{socket.gethostname()}\0{os.path.realpath(path)}"
    return hashlib.sha256(key.encode()).hexdigest()[:6]

def share_filename(path) -> str:
    "The published page's filename for a notebook: '<basename>_<slug>.html' (see share_slug)."
    return f'{Path(path).stem}_{share_slug(path)}.html'

def render_share_html(nb_path, outdir=None) -> Path:
    "Render one notebook to a single self-contained HTML file that looks like the docs site: run color_cells over its cells (the same processor the nbdev docs build uses, via doc_procs), then hand it to Quarto with the docs theme. Read and written with nbformat, NOT json.dump -- nbformat stores a cell's source as a list of lines, and writing it back as one string makes Quarto see the ::: fences as literal text instead of parsing them, so the coloured bars silently vanish. `embed-resources` inlines CSS/JS/images so the result is one portable file (a plotly figure still pulls plotly.js from a CDN, so that one element needs the network). `execute: enabled: false` guarantees sharing a notebook never runs its code."
    import nbformat
    nb_path = Path(nb_path)
    outdir = Path(outdir or tempfile.mkdtemp(prefix='boopshare-'))
    outdir.mkdir(parents=True, exist_ok=True)
    nbs = Path(__file__).parent.parent/'nbs' if '__file__' in globals() else Path('nbs')
    for asset in ('styles.css', 'booptheme.scss'):        # the docs theme this render depends on
        src = nbs/asset
        if src.exists(): shutil.copy(src, outdir/asset)
    (outdir/'_quarto.yml').write_text(_QUARTO_YML)
    stem = share_filename(nb_path)[:-len('.html')]
    doc = nbformat.read(str(nb_path), as_version=4)
    for c in doc.cells: color_cells(c)
    nbformat.write(doc, str(outdir/f'{stem}.ipynb'))
    subprocess.run(['quarto', 'render', f'{stem}.ipynb'], cwd=outdir, capture_output=True, timeout=300, check=True)
    out = outdir/f'{stem}.html'
    if not out.exists(): raise RuntimeError(f'quarto produced no output for {nb_path}')
    return out

def _gh_owner() -> str:
    "The GitHub login that owns the share repo, from the authenticated gh CLI."
    r = subprocess.run(['gh', 'api', 'user', '--jq', '.login'], capture_output=True, text=True, timeout=30, check=True)
    return r.stdout.strip()

def _ensure_share_clone(owner:str) -> Path:
    "The local clone of the share repo, cloned on first use and fast-forwarded after -- created (with Pages enabled) if the repo doesn't exist yet. Kept as a real clone rather than pushed from a temp dir each time so the history is coherent and a failed push can be retried."
    if not SHARE_CLONE.exists():
        SHARE_CLONE.parent.mkdir(parents=True, exist_ok=True)
        url = f'https://github.com/{owner}/{SHARE_REPO}.git'
        if subprocess.run(['gh', 'repo', 'view', f'{owner}/{SHARE_REPO}'], capture_output=True).returncode:
            subprocess.run(['gh', 'repo', 'create', f'{owner}/{SHARE_REPO}', '--public',
                            '-d', 'Notebooks shared from boopiter'], capture_output=True, timeout=60, check=True)
            subprocess.run(['git', 'init', '-q', '-b', 'main', str(SHARE_CLONE)], check=True, timeout=60)
            (SHARE_CLONE/'README.md').write_text('Notebooks shared from boopiter, served via GitHub Pages.\n')
            for c in (['git','add','-A'], ['git','commit','-qm','init'], ['git','remote','add','origin',url],
                      ['git','push','-q','-u','origin','main']):
                subprocess.run(c, cwd=SHARE_CLONE, check=True, timeout=120)
            subprocess.run(['gh','api','-X','POST',f'repos/{owner}/{SHARE_REPO}/pages',
                            '-f','source[branch]=main','-f','source[path]=/'], capture_output=True, timeout=60)
        else:
            subprocess.run(['git', 'clone', '-q', url, str(SHARE_CLONE)], check=True, timeout=180)
    else:
        subprocess.run(['git', 'pull', '-q', '--ff-only'], cwd=SHARE_CLONE, capture_output=True, timeout=120)
    return SHARE_CLONE

def share_notebook(nb_path) -> str:
    "Render `nb_path` and publish it to the share repo's Pages site, returning the public URL. Overwrites in place: the filename is derived from the notebook's identity (see share_slug), so re-sharing after a correction updates the page a recipient already has the link to, rather than minting a new URL. Returns as soon as the push lands -- GitHub Pages then takes roughly a minute to rebuild, so the URL 404s briefly before going live. Raises if quarto, gh or git fail; the caller surfaces that (see share_nb in cells.py)."
    html = render_share_html(nb_path)
    owner = _gh_owner()
    clone = _ensure_share_clone(owner)
    shutil.copy(html, clone/html.name)
    subprocess.run(['git', 'add', '-A'], cwd=clone, check=True, timeout=60)
    if subprocess.run(['git', 'diff', '--cached', '--quiet'], cwd=clone).returncode:  # nothing staged == unchanged notebook
        subprocess.run(['git', 'commit', '-qm', f'Share {html.name}'], cwd=clone, check=True, timeout=60)
        subprocess.run(['git', 'push', '-q'], cwd=clone, check=True, timeout=180)
    return f'https://{owner}.github.io/{SHARE_REPO}/{html.name}'